# Transforms with changing dimensionality

A particular and important kind of transforms,
are transforms that change the dimensionality of the data.
Common examples for this case are:
- 2D to 3D transforms, e.g. for aligning a 2D slice to a 3D volume
- 2D + channel to 2D transforms, e.g. for aligning a 2D slice with multiple channels to a 2D slice with a single channel

The transform that expresses this change in dimensionality is the [`ProjectAxis` transform](https://ngff.openmicroscopy.org/specifications/dev/index.html#projectaxis).

This tutorial demonstrates its usage for the case of a 2D to 3D transform,
where a 2D slice is aligned to a 3D volume.

In [ ]:
from skimage import data

from ome_zarr import OMEZarrImage, OMEZarrMultiscale, OMEZarrScene

In [2]:
img = data.cells3d().transpose((1, 0, 2, 3))
img.shape

c:\Users\johan\Documents\GitHub\ome-zarr-py\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(2, 60, 256, 256)

In [3]:
some_slice = img[0, 30, :, :]
some_slice.shape

(256, 256)

In [4]:
ngff_img = OMEZarrImage(
    data=img,
    axes=["c", "z", "y", "x"],
    scale={"c": 1, "z": 1, "y": 1, "x": 1},
    name="cells3d"
)

ngff_ms = OMEZarrMultiscale(
    image=ngff_img,
)

slice_img = OMEZarrImage(
    data=some_slice,
    axes=["y", "x"],
    scale={"y": 1, "x": 1},
    name="cells3d_slice"
)

slice_ms = OMEZarrMultiscale(
    image=slice_img,
)

In [5]:
transform_to_3d = {
    "type": "sequence",
    "input": {"path": "cells3d_slice", "name": "physical"},
    "output": {"path": "cells3d", "name": "physical"},
    "transformations": [
        {
            "type": "projectAxis",
            "createdOutputs": [0, 1]
        },
        {
            "type": "translation",
            "translation": [0, 30, 0, 0]
        }
    ]
}

In [6]:
scene = OMEZarrScene(
    images=[ngff_ms, slice_ms],
    coordinate_transformations=[transform_to_3d]
)

In [7]:
scene.to_ome_zarr("scene_2d_to_3d.zarr", overwrite=True)

Writing images: 100%|██████████| 2/2 [00:01<00:00,  1.98it/s]
